In [2]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

# 1. Cargar la llave de forma segura desde el archivo .env
load_dotenv()
API_KEY = os.getenv('YOUTUBE_API_KEY')
VIDEO_ID = 'dQw4w9WgXcQ'  

# 2. Definir la función que jala los datos
def obtener_datos_rapidos(video_id, api_key):
    print(f"Consultando a la API para el video: {video_id}...")
    
    url_stats = f"https://www.googleapis.com/youtube/v3/videos?part=statistics,snippet&id={video_id}&key={api_key}"
    res_stats = requests.get(url_stats).json()
    
    url_comments = f"https://www.googleapis.com/youtube/v3/commentThreads?part=snippet&videoId={video_id}&key={api_key}&maxResults=50"
    res_comments = requests.get(url_comments).json()
    
    if not res_stats.get('items'):
        return "Video no encontrado.", None
        
    item_stats = res_stats['items'][0]
    df_stats = pd.DataFrame([{
        'video_id': video_id,
        'titulo': item_stats['snippet']['title'],
        'vistas': int(item_stats['statistics'].get('viewCount', 0)),
        'likes': int(item_stats['statistics'].get('likeCount', 0)),
        'comentarios': int(item_stats['statistics'].get('commentCount', 0))
    }])
    
    lista_comentarios = []
    if res_comments.get('items'):
        for item in res_comments['items']:
            comentario = item['snippet']['topLevelComment']['snippet']
            lista_comentarios.append({
                'texto': comentario['textOriginal'],
                'likes_comentario': comentario['likeCount']
            })
    
    return df_stats, pd.DataFrame(lista_comentarios)

# 3. Ejecutar la extracción y mostrar las tablas
df_video, df_comentarios = obtener_datos_rapidos(VIDEO_ID, API_KEY)
display(df_video)
display(df_comentarios.head())

import os
# Crear la carpeta si no existe
os.makedirs('../data/raw', exist_ok=True)

# Guardar los datos sin el índice
df_video.to_csv('../data/raw/video_stats_sample.csv', index=False)
df_comentarios.to_csv('../data/raw/comentarios_sample.csv', index=False)
print("¡Datos crudos guardados en data/raw/!")

Consultando a la API para el video: dQw4w9WgXcQ...


,video_id,titulo,vistas,likes,comentarios
0,dQw4w9WgXcQ,Rick Astley - Never Gonna Give You Up (Officia...,1815811012,19391464,2457178


,texto,likes_comentario
0,can confirm: he never gave us up,315628
1,FINALLY I FOUND THIS SOUND 😭❤️‍🩹🔥,0
2,Hahahahaha you rickrolled me,0
3,I loved this song,0
4,Thx for the yt link btw and I’m still a alien,1


¡Datos crudos guardados en data/raw/!
